# Mixture of Experts and Reasoning Models

**Module 16** — The Modern LLM Stack

> From the [Zylo Learning Platform](https://socrapy-local.vercel.app) Sequence Models course.


## Context

The idea: replace the single FFN in each Transformer block with **multiple expert FFNs**. A learned **router** network examines each token and selects the top-k experts (typically k=2). Only the selected experts are activated; the rest are skipped.


## Code


In [ ]:
# Mixture of Experts (conceptual)
class MoELayer(nn.Module):
    def __init__(self, d_model, d_ff, n_experts, top_k=2):
        super().__init__()
        self.experts = nn.ModuleList([
            nn.Sequential(nn.Linear(d_model, d_ff), nn.ReLU(), nn.Linear(d_ff, d_model))
            for _ in range(n_experts)
        ])
        self.router = nn.Linear(d_model, n_experts)  # predicts expert scores
        self.top_k = top_k
    
    def forward(self, x):
        # Router decides which experts to use for each token
        router_logits = self.router(x)  # (B, T, n_experts)
        top_k_scores, top_k_indices = router_logits.topk(self.top_k, dim=-1)
        top_k_weights = F.softmax(top_k_scores, dim=-1)
        
        # Only compute selected experts (sparse activation)
        output = torch.zeros_like(x)
        for i in range(self.top_k):
            expert_idx = top_k_indices[..., i]
            weight = top_k_weights[..., i:i+1]
            # Each token routes to its chosen expert
            expert_out = self.experts[expert_idx](x)  # simplified
            output += weight * expert_out
        return output

# Mixtral 8x7B: 8 experts of 7B each = 47B total params
# But only 2 experts active per token = 13B active params
# Nearly GPT-3.5 quality at 1/13th the inference cost!

## Key Takeaway

**Mixtral 8x7B** (Mistral, 2023): 8 expert FFNs, 2 active per token. 47B total parameters, but only 13B are used per token. Matches GPT-3.5 quality at a fraction of the cost. This is MoE's breakthrough moment.

---

*Generated from the [Zylo Sequence Models Course](https://socrapy-local.vercel.app). Continue learning at the platform for interactive exercises, quizzes, and AI tutoring.*
